## General lab instructions
For every lab in this course, you must submit **two items**:

### **1. A Jupyter Notebook (`.ipynb`)**

Your notebook should:

* Contain **all code** needed to run your experiments
* Produce **all figures, tables, and outputs** used in your report
* Be **fully runnable from top to bottom** on a fresh kernel
* Include clear comments explaining your reasoning and workflow

The notebook will be evaluated both for **correctness** and **clarity**.

---

### **2. A Formal Lab Report (PDF or Markdown-to-PDF)**

Your report should:

* Be **self-contained** — a reader should understand everything without opening your notebook
* Present results in a **clean, professional style**
* Include:

  * All asked-for figures with captions
  * All asked-for tables summarizing performance
  * Answers to all discussion questions 


---

### 🔍 Academic Expectations

You must be prepared to:

* **Explain your code**
* **Walk through your debugging steps**
* **Justify your modeling decisions**

Some future quizzes may include questions **directly about the code you were supposed to write**, so make sure you understand every part of your implementation.



# Lab 2: Implementing Logistic Regression and Custom Neural Components

In this lab, you will build key components of modern machine learning models **from scratch**. You will begin by implementing a full multiclass logistic regression model—including the **loss layer**—and training it using both SGD and Adam. You will then extend these ideas to create your own multilayer perceptron (MLP) using custom modules.  

 
---

## What You Will Implement

### 🔹 Part 1 — Multiclass Logistic Regression (Fully Implemented by You)

You will write a complete multiclass logistic regression model, including:

* **Linear forward pass**
  ( z = XW + b )
* **Softmax function**
  ( \text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}} )
* **Cross-entropy loss (forward)**
  ( L = -\log(\text{softmax}(z)_{y}) )
* **Backward pass for the entire loss layer**
  Derive and implement
  [
  \frac{\partial L}{\partial z} = \text{softmax}(z) - \text{one_hot}(y)
  ]
* **Parameter updates using:**

  * **SGD** that you implement
  * **Adam** that you also implement (not PyTorch’s version)

You will train your model on:

* **Linearly separable blobs**
* **Overlapping ellipsoids**
* **Spiral dataset**

You will visualize all decision boundaries and compare performance across datasets and optimizers.

---

## 🔹 Part 2 — Building Your Own MLP

You will extend the ideas from Part 1 to create a multilayer neural network.

You will:

* Use the provided **Linear layer** as a reference
* Implement your own **ReLU activation** (forward + backward)
* Assemble these into an MLP architecture of your choice
* Train the MLP using your own:

  * **SGD optimizer**
  * **Adam optimizer**
* Reuse your **loss layer implementation** from Part 1
  (your MLP should call your loss code, not PyTorch’s)

You must demonstrate:

* Different depths
* Different widths
* Stability of training
* Comparison of SGD vs Adam

You will again train on the three datasets and record results in tables.

 


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import math
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

import numpy as np
import matplotlib.pyplot as plt


## Visualizing Three Types of Separable Datasets

In this section, we compare **three multiclass datasets** that are all separable, but in fundamentally different ways. Each dataset highlights a different level of geometric complexity and poses different challenges for classical machine learning models.

### **1. Linearly Separable Blobs**

A simple dataset where each class forms a compact, roughly spherical cluster.
The classes can be separated using straight lines (linear decision boundaries).
Models like **logistic regression**, **decision trees**, and **KNN** all perform very well here.

---

### **2. Overlapping Ellipsoids**

A moderately challenging dataset where each class forms an **ellipse** with its own size, orientation, and eccentricity.
The ellipsoids **overlap**, so the dataset is still separable but not perfectly clean.
Linear models begin to struggle, while models that capture more structure—like **KNN**, **decision trees**, and small **MLPs**—perform better.

---

### **3. Spiral Dataset**

A highly nonlinear dataset in which each class forms a spiral arm.
These classes are separable **only through very complex, curved boundaries**.
Linear models fail completely, and even classical nonlinear methods (trees, KNN) perform inconsistently.
Only models capable of deep nonlinear transformations—such as multilayer perceptrons (MLPs)—can learn an effective decision boundary.


In [ ]:
import torch
import math
import matplotlib.pyplot as plt

# ----------------------------
# Linearly separable blobs
# ----------------------------
def make_separable_blobs(n_points_per_class=100, n_classes=3, spread=0.5, seed=0):
    torch.manual_seed(seed)
    X = torch.zeros(n_points_per_class * n_classes, 2)
    y = torch.zeros(n_points_per_class * n_classes, dtype=torch.long)

    # Place class centers evenly around a circle
    angles = torch.linspace(0, 2 * math.pi, n_classes + 1)[:-1]
    centers = torch.stack([torch.cos(angles), torch.sin(angles)], dim=1) * 3.0

    for j in range(n_classes):
        ix = range(j * n_points_per_class, (j + 1) * n_points_per_class)
        center = centers[j]

        # Gaussian around each center
        X[ix] = center + torch.randn(n_points_per_class, 2) * spread
        y[ix] = j

    return X, y


# ----------------------------
# Spiral dataset
# ----------------------------
def make_spiral(n_points_per_class=100, n_classes=3, noise=0.2, seed=1):
    torch.manual_seed(seed)
    X = torch.zeros(n_points_per_class * n_classes, 2)
    y = torch.zeros(n_points_per_class * n_classes, dtype=torch.long)

    for j in range(n_classes):
        ix = range(n_points_per_class * j, n_points_per_class * (j + 1))
        r = torch.linspace(0.0, 1, n_points_per_class)
        t = torch.linspace(j * 4, (j + 1) * 4, n_points_per_class) + torch.randn(n_points_per_class) * noise
        X[ix] = torch.stack((r * torch.sin(t), r * torch.cos(t)), dim=1)
        y[ix] = j

    return X, y


# ----------------------------
# Overlapping ellipsoids
# ----------------------------
def make_ellipsoids(n_points_per_class=100, n_classes=3, scale=1.0, seed=2):
    """
    Creates overlapping ellipsoidal clusters with different shapes,
    sizes, and orientations. The centers are intentionally placed close 
    together to guarantee overlap.
    """
    torch.manual_seed(seed)

    X = torch.zeros(n_points_per_class * n_classes, 2)
    y = torch.zeros(n_points_per_class * n_classes, dtype=torch.long)

    # Centers closer together (radius = 1.2) to force overlap
    angles = torch.linspace(0, 2 * math.pi, n_classes + 1)[:-1]
    centers = torch.stack([torch.cos(angles), torch.sin(angles)], dim=1) * 1.2

    for j in range(n_classes):
        ix = range(j * n_points_per_class, (j + 1) * n_points_per_class)
        center = centers[j]

        # Eccentricity: random but consistent
        eigvals = torch.tensor([
            torch.rand(1).item() * 0.8 + 0.2,  # major axis
            torch.rand(1).item() * 0.2 + 0.05  # minor axis
        ]) * scale

        # Random orientation
        theta = torch.rand(1).item() * math.pi
        R = torch.tensor([[math.cos(theta), -math.sin(theta)],
                          [math.sin(theta),  math.cos(theta)]])

        cov = R @ torch.diag(eigvals) @ R.t()

        # Sample from ellipsoid
        pts = torch.distributions.MultivariateNormal(center, cov).sample((n_points_per_class,))
        X[ix] = pts
        y[ix] = j

    return X, y


# ----------------------------
# Generate all three datasets
# ----------------------------
X_sep,   y_sep   = make_separable_blobs(n_classes=3, spread=0.4)
X_ellip, y_ellip = make_ellipsoids(n_classes=3, scale=1.0)
X_spiral,y_spiral = make_spiral()


# ----------------------------
# Combined plot
# ----------------------------
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.scatter(X_sep[:, 0], X_sep[:, 1], c=y_sep, cmap="viridis", s=15)
plt.title("Linearly Separable Blobs")
plt.xlabel("x1"); plt.ylabel("x2")

plt.subplot(1, 3, 2)
plt.scatter(X_ellip[:, 0], X_ellip[:, 1], c=y_ellip, cmap="viridis", s=15)
plt.title("Overlapping Ellipsoids")
plt.xlabel("x1"); plt.ylabel("x2")

plt.subplot(1, 3, 3)
plt.scatter(X_spiral[:, 0], X_spiral[:, 1], c=y_spiral, cmap="viridis", s=15)
plt.title("Spiral Dataset")
plt.xlabel("x1"); plt.ylabel("x2")

plt.tight_layout()
plt.show()


# Part 1 — Implementing Your Own Multiclass Logistic Regression

In this part of the lab, you will **implement multiclass logistic regression from scratch** using only **NumPy**.
You may consult ideas from the previous lab, but **all code inside your class must be written by you and you must understand how every line works**.

Your implementation must include the following components:

### Core functions (must be written by you)

1. `crossentropy_loss(logits, y)`
2. `crossentropy_grad(X, y, logits)`
3. `fit(X, y, method="sgd")` — trains the model using either SGD or Adam
4. `predict(X)` — returns predicted class labels

### Optimizer functions (also must be written by you)

Inside your class, you will add:

5. `_sgd_update(grad_W, grad_b)`
6. `_adam_update(grad_W, grad_b, t)`

These will be called internally by `fit(...)` depending on the `method` argument.
**You are responsible for implementing both update rules.**

You will then train and evaluate your model on:

1. **Linearly separable blobs**
2. **Overlapping ellipsoids**
3. **Spiral dataset**

Since the plot code is quite complicated, I have given you mine. You may use it, or adapt it, or totally rewrite it, to fit your own needs. If you do use mine, you should understand it well -- I may have you implement plotting pseudocode in a quiz.

---

## Academic Responsibility

Everything you implement in `MyMulticlassLogReg` **must be your own work**.

You will later be expected to answer quiz questions about this code, including:

* debugging
* reasoning about gradients
* comparing optimizers
* writing pseudocode

If you cannot explain your code, you did not really write it.

---

# Deliverables for Part 1

You must submit the following items in your lab notebook and your formal write-up.

---

### **1. Your completed `MyMulticlassLogReg` class**

All six functions must be implemented and working:

* `crossentropy_loss`
* `crossentropy_grad`
* `_sgd_update`
* `_adam_update`
* `fit`
* `predict`

---

### **2. Training curves for all three datasets**

For each dataset (Blobs, Ellipsoids, Spirals), you must produce:

* **One plot showing loss vs. iteration**

  * Overlay **SGD** and **Adam** on the same axes
  * 
* **One plot showing training misclassification vs. iteration**

  * Again overlay **SGD** and **Adam**

This results in **6 total plots**, or **3 pairs** of overlaid curves.

**Important:**
All figures must have **descriptive and interpretive captions**, not generic ones.
Your caption must explain **what the curves show** and **why the two optimizers behave differently**.

---

### **3. A 1×3 Decision-Boundary Figure**

Same format as Lab 1:

* Column 1: blobs
* Column 2: ellipsoids
* Column 3: spirals

Each panel must include:

* decision boundary visualization
* training points
* the **final misclassification rate** in the title
* a caption interpreting the boundary (e.g., linear vs nonlinear shape)

---
 

### **4. Discussion Questions**

Include short, thoughtful answers:

1. Compare the computational cost of model evaluation (logits → probabilities → predictions) with the cost of computing gradients. Which steps dominate and why?

2. Could you reduce computation by combining the loss and gradient calculations? What redundant steps are repeated in your current implementation, and how might you reorganize the code to avoid them?

3. Compare the convergence behavior of SGD and Adam across the three datasets. Which optimizer converged faster, and how does dataset geometry (linear vs nonlinear vs highly entangled) influence the relative performance of the two methods? Be specific and refer to your training curves.

4. Do you notice any difference in the training behavior between the different types of problems? What could account for them?
 


In [ ]:
def plot_decision_boundary(X, y, model, title="", ax=None):
    """
    Visualize the classifier decision boundary on a 2D feature space.

    A decision boundary is the set of points where the model is indifferent
    between classes (the predicted class changes across the boundary). In the
    contour plot, background colors show the class predicted at each location
    in the plane, while the overlaid points show the true training samples
    colored by their labels.

    In y = f(x) terms for this function, x is a 2D input feature vector
    [x1, x2] and y is the predicted discrete class index.
    """
    
    h = 0.01
    x_min, x_max = X[:, 0].min() - 0.2, X[:, 0].max() + 0.2
    y_min, y_max = X[:, 1].min() - 0.2, X[:, 1].max() + 0.2

    xx, yy = np.meshgrid(
        np.arange(x_min, x_max, h),
        np.arange(y_min, y_max, h)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]

    Z = model.predict(grid).reshape(xx.shape)

    if ax is None:
        ax = plt.gca()

    ax.contourf(xx, yy, Z, cmap="Spectral", alpha=0.4)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="Spectral", edgecolor="k", s=20)
    ax.set_title(title)


def plot_all_training_curves(loss_ax,err_ax,    loss_sgd, err_sgd, loss_adam, err_adam, probtype):


    # ----------------------
    # Loss curve
    # ----------------------
    loss_ax.plot(loss_sgd,  label="SGD",  linestyle="-")
    loss_ax.plot(loss_adam, label="Adam", linestyle="--")
    loss_ax.set_title(f"{probtype}: Loss")
    loss_ax.set_xlabel("Iteration")
    loss_ax.set_ylabel("Loss")
    loss_ax.legend()

    # ----------------------
    # Misclassification curve
    # ----------------------
    err_ax.plot(err_sgd,  label="SGD",  linestyle="-")
    err_ax.plot(err_adam, label="Adam", linestyle="--")
    err_ax.set_title(f"{probtype}: Misclassification")
    err_ax.set_xlabel("Iteration")
    err_ax.set_ylabel("Error rate")
    err_ax.legend()

 


In [ ]:
import numpy as np

class MyMulticlassLogReg:
    def __init__(self, lr=1e-1, max_iter=500, method="sgd"):
        self.lr = lr
        self.max_iter = max_iter
        self.method = method.lower()
        if self.method not in {"sgd", "adam"}:
            raise ValueError("method must be either 'sgd' or 'adam'")

        self.W = None
        self.b = None

        # Adam state
        self.beta1 = 0.9
        self.beta2 = 0.999
        self.eps = 1e-8
        self.mW = None
        self.vW = None
        self.mb = None
        self.vb = None

    # ------------------------------------------------------------
    # Cross-entropy loss
    # ------------------------------------------------------------
    def crossentropy_loss(self, logits, y):
        shifted = logits - logits.max(axis=1, keepdims=True)
        logsumexp = np.log(np.exp(shifted).sum(axis=1, keepdims=True))
        log_probs = shifted - logsumexp
        return -np.mean(log_probs[np.arange(len(y)), y])

    # ------------------------------------------------------------
    # Gradient of CE wrt W, b
    # ------------------------------------------------------------
    def crossentropy_grad(self, X, y, logits):
        shifted = logits - logits.max(axis=1, keepdims=True)
        exp_scores = np.exp(shifted)
        probs = exp_scores / exp_scores.sum(axis=1, keepdims=True)

        n = X.shape[0]
        probs[np.arange(n), y] -= 1
        probs /= n

        grad_W = X.T @ probs
        grad_b = probs.sum(axis=0)
        return grad_W, grad_b

    # ------------------------------------------------------------
    # SGD update
    # ------------------------------------------------------------
    def _sgd_update(self, grad_W, grad_b):
        self.W -= self.lr * grad_W
        self.b -= self.lr * grad_b

    # ------------------------------------------------------------
    # Adam update
    # ------------------------------------------------------------
    def _adam_update(self, grad_W, grad_b, t):
        self.mW = self.beta1 * self.mW + (1 - self.beta1) * grad_W
        self.vW = self.beta2 * self.vW + (1 - self.beta2) * (grad_W ** 2)
        self.mb = self.beta1 * self.mb + (1 - self.beta1) * grad_b
        self.vb = self.beta2 * self.vb + (1 - self.beta2) * (grad_b ** 2)

        mW_hat = self.mW / (1 - self.beta1 ** t)
        vW_hat = self.vW / (1 - self.beta2 ** t)
        mb_hat = self.mb / (1 - self.beta1 ** t)
        vb_hat = self.vb / (1 - self.beta2 ** t)

        self.W -= self.lr * mW_hat / (np.sqrt(vW_hat) + self.eps)
        self.b -= self.lr * mb_hat / (np.sqrt(vb_hat) + self.eps)

    # ------------------------------------------------------------
    # Fit model + return training curves
    # ------------------------------------------------------------
    def fit(self, X, y):
        n_samples, n_features = X.shape
        n_classes = int(np.max(y)) + 1

        rng = np.random.default_rng(0)
        self.W = 0.01 * rng.standard_normal((n_features, n_classes))
        self.b = np.zeros(n_classes)

        self.mW = np.zeros_like(self.W)
        self.vW = np.zeros_like(self.W)
        self.mb = np.zeros_like(self.b)
        self.vb = np.zeros_like(self.b)

        loss_history = []
        err_history = []

        for t in range(1, self.max_iter + 1):
            logits = X @ self.W + self.b

            loss = self.crossentropy_loss(logits, y)
            preds = np.argmax(logits, axis=1)
            err = np.mean(preds != y)

            loss_history.append(loss)
            err_history.append(err)

            grad_W, grad_b = self.crossentropy_grad(X, y, logits)

            if self.method == "sgd":
                self._sgd_update(grad_W, grad_b)
            else:
                self._adam_update(grad_W, grad_b, t)

        return loss_history, err_history

    # ------------------------------------------------------------
    # Predict
    # ------------------------------------------------------------
    def predict(self, X):
        logits = X @ self.W + self.b
        shifted = logits - logits.max(axis=1, keepdims=True)
        exp_scores = np.exp(shifted)
        probs = exp_scores / exp_scores.sum(axis=1, keepdims=True)
        return np.argmax(probs, axis=1)


In [ ]:
# ============================================================
# Convert Torch → NumPy
# ============================================================
Xs, ys     = X_sep.numpy(),     y_sep.numpy()
Xell, yell = X_ellip.numpy(),   y_ellip.numpy()
Xsp, ysp   = X_spiral.numpy(),  y_spiral.numpy()

# ============================================================
# Decision boundaries (1×3)
# Training curves (1×3)
# ============================================================
fig1, axes1 = plt.subplots(2, 3, figsize=(12, 6))
fig2, axes2 = plt.subplots(2, 3, figsize=(12, 6))


# ============================================================
# BLOBS
# ============================================================
# ---- SGD ----
log_sgd = MyMulticlassLogReg(lr=0.1, max_iter=500, method="sgd")
loss_sgd_blob, err_sgd_blob = log_sgd.fit(Xs, ys)
pred_sgd  = log_sgd.predict(Xs)
mis_sgd   = (pred_sgd != ys).mean()
plot_decision_boundary(
    Xs, ys, log_sgd,
    title=f"Blobs (SGD, err={mis_sgd:.2f})",
    ax=axes1[0,0]
)

# ---- Adam ----
log_adam = MyMulticlassLogReg(lr=0.1, max_iter=500, method="adam")
loss_adam_blob, err_adam_blob = log_adam.fit(Xs, ys)
pred_adam = log_adam.predict(Xs)
mis_adam  = (pred_adam != ys).mean()
plot_decision_boundary(
    Xs, ys, log_adam,
    title=f"Blobs (Adam, err={mis_adam:.2f})",
    ax=axes1[1,0]
)

# ---- Training curves ----
plot_all_training_curves(
    axes2[0,0], axes2[1,0],
    loss_sgd_blob, err_sgd_blob,
    loss_adam_blob, err_adam_blob,
    probtype="Blobs"
)


# ============================================================
# ELLIPSOIDS
# ============================================================
# ---- SGD ----
log_sgd = MyMulticlassLogReg(lr=0.1, max_iter=500, method="sgd")
loss_sgd_ell, err_sgd_ell = log_sgd.fit(Xell, yell)
pred_sgd = log_sgd.predict(Xell)
mis_sgd  = (pred_sgd != yell).mean()
plot_decision_boundary(
    Xell, yell, log_sgd,
    title=f"Ellipsoids (SGD, err={mis_sgd:.2f})",
    ax=axes1[0,1]
)

# ---- Adam ----
log_adam = MyMulticlassLogReg(lr=0.1, max_iter=500, method="adam")
loss_adam_ell, err_adam_ell = log_adam.fit(Xell, yell)
pred_adam = log_adam.predict(Xell)
mis_adam  = (pred_adam != yell).mean()
plot_decision_boundary(
    Xell, yell, log_adam,
    title=f"Ellipsoids (Adam, err={mis_adam:.2f})",
    ax=axes1[1,1]
)

# ---- Training curves ----
plot_all_training_curves(
    axes2[0,1], axes2[1,1],
    loss_sgd_ell, err_sgd_ell,
    loss_adam_ell, err_adam_ell,
    probtype="Ellipsoids"
)


# ============================================================
# SPIRALS
# ============================================================
# ---- SGD ----
log_sgd = MyMulticlassLogReg(lr=0.1, max_iter=500, method="sgd")
loss_sgd_spi, err_sgd_spi = log_sgd.fit(Xsp, ysp)
pred_sgd = log_sgd.predict(Xsp)
mis_sgd  = (pred_sgd != ysp).mean()
plot_decision_boundary(
    Xsp, ysp, log_sgd,
    title=f"Spirals (SGD, err={mis_sgd:.2f})",
    ax=axes1[0,2]
)

# ---- Adam ----
log_adam = MyMulticlassLogReg(lr=0.1, max_iter=500, method="adam")
loss_adam_spi, err_adam_spi = log_adam.fit(Xsp, ysp)
pred_adam = log_adam.predict(Xsp)
mis_adam  = (pred_adam != ysp).mean()
plot_decision_boundary(
    Xsp, ysp, log_adam,
    title=f"Spirals (Adam, err={mis_adam:.2f})",
    ax=axes1[1,2]
)

# ---- Training curves ----
plot_all_training_curves(
    axes2[0,2], axes2[1,2],
    loss_sgd_spi, err_sgd_spi,
    loss_adam_spi, err_adam_spi,
    probtype="Spirals"
)


fig1.tight_layout()
fig2.tight_layout()

# -----------------------------------
# Save the figures to disk
# -----------------------------------
fig1.savefig("logreg_decisionboundaries.png", dpi=200, bbox_inches="tight")
fig2.savefig("logreg_trainingcurves.png", dpi=200, bbox_inches="tight")


